In [2]:
import fastf1

fastf1.Cache.enable_cache("../cache")

session = fastf1.get_session(2024, "Monza", "R")
session.load()

print(session.results[["Abbreviation", "GridPosition", "Position"]])

core           INFO 	Loading data for Italian Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['16', '81', '4', '55', '44', '1', '63', '11', '23', '20', '14', '43', '3', '31', '10', '77', '27', '24', '18', '22']


   Abbreviation  GridPosition  Position
16          LEC           4.0       1.0
81          PIA           2.0       2.0
4           NOR           1.0       3.0
55          SAI           5.0       4.0
44          HAM           6.0       5.0
1           VER           7.0       6.0
63          RUS           3.0       7.0
11          PER           8.0       8.0
23          ALB           9.0       9.0
20          MAG          13.0      10.0
14          ALO          11.0      11.0
43          COL          18.0      12.0
3           RIC          12.0      13.0
31          OCO          15.0      14.0
10          GAS          14.0      15.0
77          BOT          19.0      16.0
27          HUL          10.0      17.0
24          ZHO          20.0      18.0
18          STR          17.0      19.0
22          TSU          16.0      20.0


In [5]:
import plotly.express as px

# get one driver's fastest lap
lap = session.laps.pick_drivers("NOR").pick_fastest()

# get telemetry data for that lap
telemetry = lap.get_car_data().add_distance()

print(telemetry[["Distance", "Speed", "Throttle", "Brake", "nGear"]].head())

fig = px.line(telemetry, x="Distance", y="Speed", title=f"Telemetry for NOR, Monza 2024 ({lap['LapTime']})")

fig.show()

     Distance  Speed  Throttle  Brake  nGear
0   28.176667  321.0      99.0  False      8
1   42.443333  321.0      99.0  False      8
2   63.976667  323.0     100.0  False      8
3   89.254444  325.0      99.0  False      8
4  114.610000  326.0      99.0  False      8


In [6]:
import pandas as pd
import plotly.express as px

def format_laptime(lap_time):
    total = lap_time.total_seconds()
    minutes = int(total // 60)
    seconds = total - minutes * 60
    return f"{minutes}:{seconds:06.3f}"

drivers = ["NOR", "LEC"]
frames = []
labels = []

for code in drivers:
    lap = session.laps.pick_drivers(code).pick_fastest()
    telemetry = lap.get_car_data().add_distance()
    telemetry["Driver"] = code
    frames.append(telemetry)
    labels.append(f"{code} ({format_laptime(lap['LapTime'])})")

combined = pd.concat(frames)

fig = px.line(combined, x="Distance", y="Speed", color="Driver", title=f"Telemetry Comparison, Monza 2024: " + " vs ".join(labels))
fig.show()

In [ ]:
import fastf1
from fastf1 import utils
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import pandas as pd

quali = fastf1.get_session(2024, "Monza", "Q")
quali.load()

code_a, code_b = "NOR", "LEC"

q1, q2, q3 = quali.laps.split_qualifying_sessions()
parts = {"Q1": q1, "Q2": q2, "Q3": q3}

def final_quali_lap(code):
    """Return (part name, lap) for the driver's best lap in their last qualifying part."""
    row = quali.results[quali.results["Abbreviation"] == code].iloc[0]
    for name in ["Q3", "Q2", "Q1"]:
        if pd.notna(row[name]):
            return name, parts[name].pick_drivers(code).pick_fastest()

part_a, lap_a = final_quali_lap(code_a)
part_b, lap_b = final_quali_lap(code_b)

print(code_a, part_a, format_laptime(lap_a["LapTime"]))
print(code_b, part_b, format_laptime(lap_b["LapTime"]))

tel_a = lap_a.get_car_data().add_distance()
tel_b = lap_b.get_car_data().add_distance()

# delta of lap_b relative to lap_a (positive = B is behind A)
delta, ref_tel, _ = utils.delta_time(lap_a, lap_b)

fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
                    row_heights=[0.4, 0.3, 0.3], vertical_spacing=0.04,
                    subplot_titles=("Speed (km/h)", "Throttle (%)",
                                    f"Delta: {code_b} vs {code_a} (s)"))

for tel, code, lap in [(tel_a, code_a, lap_a), (tel_b, code_b, lap_b)]:
    name = f"{code} {format_laptime(lap['LapTime'])}"
    fig.add_trace(go.Scatter(x=tel["Distance"], y=tel["Speed"], name=name,
                             legendgroup=code), row=1, col=1)
    fig.add_trace(go.Scatter(x=tel["Distance"], y=tel["Throttle"], name=name,
                             legendgroup=code, showlegend=False), row=2, col=1)

fig.add_trace(go.Scatter(x=ref_tel["Distance"], y=delta, name="Delta",
                         showlegend=False), row=3, col=1)

fig.update_xaxes(title_text="Distance (m)", row=3, col=1)
fig.update_layout(height=800, title="Qualifying comparison, Monza 2024")
fig.show()

NameError: name 'quali' is not defined